In [9]:
import json
import re
import google.generativeai as genai

# ============================================================
# 1. GEMINI API KEY
# ============================================================

genai.configure(api_key="apikey")


# ============================================================
# 2. CREATE GEMINI MODEL
# ============================================================

model = genai.GenerativeModel(
    model_name="gemini-3.6-flash",
    generation_config={
        "temperature": 0,
        "response_mime_type": "application/json"
    }
)


# ============================================================
# 3. CHECK CUSTOMER RESPONSE
# ============================================================

moderation_prompt = f"""
You are a content safety classifier.

Analyze the following customer-facing response for harmful content.

Return ONLY a valid JSON object.

Use exactly this structure:

{{
    "flagged": false,
    "harassment": false,
    "hate_speech": false,
    "sexually_explicit": false,
    "dangerous_content": false,
    "violence": false,
    "reason": ""
}}

Rules:

1. "flagged" must be true if any harmful category is true.
2. "harassment" is true only if the text contains harassment, bullying,
   threats, or abusive targeting of a person.
3. "hate_speech" is true only if the text attacks a protected group.
4. "sexually_explicit" is true only if the text contains explicit sexual content.
5. "dangerous_content" is true only if the text promotes dangerous activities.
6. "violence" is true only if the text contains violent content.
7. "reason" should briefly explain the reason if flagged.
8. If the response is safe, "reason" must be an empty string.

Customer-facing response:

\"\"\"
{final_response_to_customer}
\"\"\"
"""


# ============================================================
# 4. SEND TO GEMINI
# ============================================================

try:

    response = model.generate_content(moderation_prompt)

    # Get Gemini response
    raw = response.text.strip()

    print("RAW GEMINI RESPONSE:")
    print(raw)


    # ========================================================
    # 5. CLEAN RESPONSE
    # ========================================================

    cleaned = raw

    # Remove ```json and ``` if Gemini returns them
    cleaned = re.sub(
        r"```json",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = cleaned.replace("```", "").strip()


    # ========================================================
    # 6. CONVERT RESPONSE TO JSON
    # ========================================================

    moderation_output = json.loads(cleaned)


    # ========================================================
    # 7. MAKE SURE ALL REQUIRED FIELDS EXIST
    # ========================================================

    moderation_output = {
        "flagged": bool(moderation_output.get("flagged", False)),
        "harassment": bool(moderation_output.get("harassment", False)),
        "hate_speech": bool(moderation_output.get("hate_speech", False)),
        "sexually_explicit": bool(
            moderation_output.get("sexually_explicit", False)
        ),
        "dangerous_content": bool(
            moderation_output.get("dangerous_content", False)
        ),
        "violence": bool(
            moderation_output.get("violence", False)
        ),
        "reason": moderation_output.get("reason", "")
    }


    # ========================================================
    # 8. PRINT MODERATION OUTPUT
    # ========================================================

    print("\nMODERATION OUTPUT:")
    print(
        json.dumps(
            moderation_output,
            indent=4
        )
    )


    # ========================================================
    # 9. SAVE JSON FILE
    # ========================================================

    json_file = "moderation_result.json"

    with open(
        json_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            moderation_output,
            f,
            indent=4,
            ensure_ascii=False
        )


    print("\nJSON file created:")
    print(json_file)


    # ========================================================
    # 10. CHECK WHETHER RESPONSE IS SAFE
    # ========================================================

    if moderation_output["flagged"] is False:

        print("\n✅ SAFE TO SEND TO CUSTOMER")

    else:

        print("\n🚫 BLOCKED")

        print(
            "Reason:",
            moderation_output["reason"]
        )


except json.JSONDecodeError as e:

    print("\n❌ JSON PARSING ERROR")
    print("Error:", e)

    print("\nRaw Gemini response:")
    print(raw)


except Exception as e:

    print("\n❌ ERROR")
    print(type(e).__name__, ":", e)

RAW GEMINI RESPONSE:
{
    "flagged": false,
    "harassment": false,
    "hate_speech": false,
    "sexually_explicit": false,
    "dangerous_content": false,
    "violence": false,
    "reason": ""
}

MODERATION OUTPUT:
{
    "flagged": false,
    "harassment": false,
    "hate_speech": false,
    "sexually_explicit": false,
    "dangerous_content": false,
    "violence": false,
    "reason": ""
}

JSON file created:
moderation_result.json

✅ SAFE TO SEND TO CUSTOMER


In [11]:
# ============================================================
# 1. CUSTOMER MESSAGE
# ============================================================

customer_message = "What does life is like a box of chocolates mean?"


# ============================================================
# 2. PRODUCT INFORMATION
# ============================================================

product_information = """
Life is like a box of chocolates means that life is unpredictable.
You never know what you are going to get or what will happen next.
"""


# ============================================================
# 3. AGENT RESPONSE
# ============================================================

another_response = "life is like a box of chocolates"


# ============================================================
# 4. SYSTEM MESSAGE
# ============================================================

system_message = """
You are an assistant that evaluates customer service responses.

Check whether:
1. The agent response uses the retrieved product information correctly.
2. The agent response sufficiently answers the customer's question.

Output only Y or N.
"""


# ============================================================
# 5. CREATE QUESTION-ANSWER PAIR
# ============================================================

q_a_pair = f"""
Customer message:
{customer_message}

Product information:
{product_information}

Agent response:
{another_response}

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""


# ============================================================
# 6. CREATE MESSAGES
# ============================================================

messages = [
    {
        "role": "system",
        "content": system_message
    },
    {
        "role": "user",
        "content": q_a_pair
    }
]


# ============================================================
# 7. GET EVALUATION
# ============================================================

response = get_completion_from_messages(messages)

print(response)

Does the response use the retrieved information correctly? N
Does the response sufficiently answer the question? N
